# Recomendation System

In [1]:
# Imports

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import random

import ast
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load the data

books_df = pd.read_csv('data/books_reduced.csv')
books_limited_df = pd.read_csv('data/books_reduced_limited.csv')

authors_df = pd.read_csv('data/authors_reduced.csv')
authors_to_books_df = pd.read_csv('data/authors_to_books_reduced.csv')
genres_df = pd.read_csv('data/genres_reduced.csv')

In [10]:
# reviews_df = pd.read_csv('data/reviews_reduced.csv')

In [ ]:
# interactions_df = pd.read_csv('data/interactions_reduced.csv')

In [3]:
print(f'size of books_df: {books_df.shape}')
print(f'size of books_limited_df: {books_limited_df.shape}')

print(f'size of authors_df: {authors_df.shape}')
print(f'size of authors_to_books_df: {authors_to_books_df.shape}')
print(f'size of genres_df: {genres_df.shape}')

# print(f'size of reviews_df: {reviews_df.shape}')

# print(f'size of interactions_df: {interactions_df.shape}')

size of books_df: (24597, 26)
size of books_limited_df: (24597, 9)
size of authors_df: (10033, 5)
size of authors_to_books_df: (34729, 3)
size of genres_df: (165977, 2)


# Plans

So, what we have available to use: we have books, which includes information 

## Let's start with the collumns in books: 

books_limited:
- 'text_reviews_count'    
- 'average_rating'  
- 'similar_books' (with causion as the books are stored in a list)  
- 'num_pages' (may only be used at the end as a final filter, similar to description)  
- 'publication_year'  
- **'book_id'**  (primary key)
- 'ratings_count'  
- 'title' (to see which books are discussed)  
- 'title_without_series' (to see which books are discussed)  

books:
- 'isbn'  
- 'series'  
- 'popular_shelves'  
- 'asin'  
- 'kindle_asin'  
- 'description' 
- 'format'  
- 'link' 
- 'publisher'  
- 'publication_day'  
- 'isbn13'  
- 'publication_month'  
- 'edition_information'  
- 'url'  
- 'image_url'  
- 'work_id'

We can filter books by the collumns in books_limited
- 'text_reviews_count': Used to show how popular a book is
- 'average_rating': Give books with higher ratings a higher score
- 'similar_books': I may use this, but I mgiht also prefer to find out that info myself. I will see if I want to create another database 'similar books' to use for this. 
- 'num_pages': User can spacify the page length they are looking for if they want
- 'publication_year': User can specify the year they want books to read from 
- **'book_id'**  (primary key)
- 'ratings_count': Used to show how popular a book is
- 'title' (to see which books are discussed)  
- 'title_without_series' (to see which books are discussed)  

So text_reviews_count, average_rating, similar_books, num_pages, publication_year, ratings_count can all be used to help filter what books o recomend

Authors and genres are self explainitory, they include information about the genre. In use, we can use the genre as a filter to only give us recomendations within the specified genre. For authors, if the user has given authors they like, we can maybe give a higher score to the books those authors have written. 

### Filter method
The values I will use to filter results include: num_pages, publication_year, genres  
The values I can use in the learning algorithm itself include: text_reviews_count, average_rating, similar_books, ratings_count, authors  
values that apply to all reguardless are: text_reviews_count, average_rating, ratings_count  
values that can impact the score depending onw hat the user is looking for: similar_books, authors  

So, right now I am thinking that since there are so many chategories I will assign each book a value and save that to a matrix. the initial valuse can be made up of text_reviews_count, average_rating, ratings_count. Then the user can specify what they are looking for, for example they give a genre and any books belonging to a genre or author the person likes gets a higher score. Same for books belonging to similar books to what the user has said they like, those books get a higher score. The user can also specify a prefered page length and for books wihtin that length they will recive a better score and books outside the length may recive a proportionally lower score that way we aren't eliminating books that are 2 pages shorter than what the user wanted. Same for publication year. 

So essentiall I will assign each book a weight and update that weight based on what the user specifies they are looking for. They can give multiple books, genres, and authors they like and specify other factors to factor into this descision. Then I will get a final ranking for each of the books and from there books with the highest ranking can be recomended. 

So that is a simple recomendation system without any more complicated parts. 

But then I also have the reviews and interactions databases. I am not sure how I want to include those, but to begin with before I make it complicated I think it may be best to start off with my initial idea for things besides that. I can build a simple system and make a user interface for that. Then I can go back and possibly add in the other 2 databases. 

# The plan

Ok, so now that I know what I want, here is the plan:
1. Create a database for similar books
2. Create new dataframe? some data structure containing book id and weight
3. Then begin updating the weights by their raitngs, ratings_count, and reviews_count
4. Then create functions to update the weight based on user input so the user can input specified valuses and the weights will be updated and the top 10 (user can specify how many) books will be shown with their descriptions and tiles and other stats (year, page number, rating, etc.) 
5. Then I want to create a user interface either online or right here for the user to have a simple way to read their results
6. Then when I and happy with my results and interface I can begin looking at reviews and interactions to get more sophisticated methods 

# Let's get started! 
Phase 1: make the database fore similar_books

In [4]:
# Let's start
# First step is to create a new database for similar_books
# Let's look at books

books_limited_df.head()

,book_id,title,num_pages,publication_year,average_rating,text_reviews_count,ratings_count,title_without_series,similar_books
0,780917,"The Serpent and the Rose (War of the Rose, #1)",320.0,2007.0,3.49,54,372,"The Serpent and the Rose (War of the Rose, #1)","['1616595', '2356304', '5432871', '9766146', '..."
1,1488663,The Greatest Gift: The Original Story That Ins...,80.0,1996.0,4.05,74,226,The Greatest Gift: The Original Story That Ins...,"['17544', '9442983', '13586799', '1755629', '6..."
2,89588,The Visitor,512.0,2003.0,3.81,61,1480,The Visitor,"['114555', '219231', '455901', '102888', '6421..."
3,89583,Six Moon Dance,544.0,1999.0,3.75,58,1321,Six Moon Dance,"['673024', '6747831', '6056679', '121611', '65..."
4,2741850,Personal Demon (Women of the Otherworld #8),523.0,2008.0,4.07,39,385,Personal Demon (Women of the Otherworld #8),"['6547188', '4894646', '1704159', '28650', '66..."


In [5]:
# Create similar_books_df

similar_books_df = books_limited_df[['book_id', 'similar_books']].copy()

# Convert the string representation of the list into an actual Python list
similar_books_df['similar_books'] = similar_books_df['similar_books'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Create one row per similar book
similar_books_df = similar_books_df.explode('similar_books')

# Rename the column
similar_books_df = similar_books_df.rename(
    columns={'similar_books': 'similar_book_id'}
)

# Remove rows where there were no similar books
similar_books_df = similar_books_df.dropna(subset=['similar_book_id'])

# Convert IDs to integers if appropriate
similar_books_df['book_id'] = similar_books_df['book_id'].astype(int)
similar_books_df['similar_book_id'] = similar_books_df['similar_book_id'].astype(int)

# Reset index
similar_books_df = similar_books_df.reset_index(drop=True)

In [6]:
similar_books_df.head()

,book_id,similar_book_id
0,780917,1616595
1,780917,2356304
2,780917,5432871
3,780917,9766146
4,780917,6058757


# Next Step!

Ok, great. Now, next up we want to create a new df to use as a matrix for the recomendation system. 

rows will include: book_id, recomendation_score, num_pages, publication_year

recomendation score as a baseline will be calculated based on text_reviews_count, average_rating, ratings_count, perhaps the average of ratings_count and text_reviews_count * average rating. This will be the baseline score, based on how popular it is and how highly it is rated.  

The next values which will be included in the matrix about each book include: num_pages, publication_year

Then there will be 3 additional matrices including genres, authors, and similar_books where we can look up information about a book to see which books can get a higher score. 

So these are the primary ways I will calculate the recomendation score. 

Now, one last thing I need to decide is how much weight to give each score. 

lets see, base rating should be pretty important as you wouldn't want to read a terrible book just because its in the right genre

num_pages and publication_year should also be ranked as if not more important as if the user specifies this then the books must match the criteria

genres, authors, and similar_books are also extremely important as this just what the user is looking for. authors may be the least important though. so i might give authors a lower score or i might just give them all the same score range, either between 

In [7]:
recommendations_df = books_df[['book_id', 'num_pages', 'publication_year', 'similar_books']].copy()

recommendations_df.insert(1, 'recommendation_score', None)

recommendations_df.head()

,book_id,recommendation_score,num_pages,publication_year,similar_books
0,780917,None,320.0,2007.0,"['1616595', '2356304', '5432871', '9766146', '..."
1,1488663,None,80.0,1996.0,"['17544', '9442983', '13586799', '1755629', '6..."
2,89588,None,512.0,2003.0,"['114555', '219231', '455901', '102888', '6421..."
3,89583,None,544.0,1999.0,"['673024', '6747831', '6056679', '121611', '65..."
4,2741850,None,523.0,2008.0,"['6547188', '4894646', '1704159', '28650', '66..."


In [8]:
# chat gpt:

normalized_scores_df = books_df[
    ['ratings_count', 'text_reviews_count']
].copy()

# Reduce the effect of extreme popularity values
normalized_scores_df['ratings_count'] = np.log1p(
    normalized_scores_df['ratings_count']
)

normalized_scores_df['text_reviews_count'] = np.log1p(
    normalized_scores_df['text_reviews_count']
)

# Scale both variables between 0 and 1
scaler = MinMaxScaler()

normalized_scores_df[
    ['ratings_count', 'text_reviews_count']
] = scaler.fit_transform(
    normalized_scores_df[
        ['ratings_count', 'text_reviews_count']
    ]
)

# Equal weighting
normalized_scores_df['popularity_score'] = (
    normalized_scores_df['ratings_count'] +
    normalized_scores_df['text_reviews_count']
) / 2

In [9]:
recommendations_df['recommendation_score'] = (
    (normalized_scores_df['popularity_score'] +
    books_df['average_rating'] / 5)
) / 2

In [10]:
normalized_scores_df.head()

,ratings_count,text_reviews_count,popularity_score
0,0.172448,0.068123,0.120285
1,0.128982,0.107134,0.118058
2,0.293132,0.083191,0.188162
3,0.283192,0.076953,0.180073
4,0.175447,0.028067,0.101757


In [11]:
recommendations_df.head()

,book_id,recommendation_score,num_pages,publication_year,similar_books
0,780917,0.409143,320.0,2007.0,"['1616595', '2356304', '5432871', '9766146', '..."
1,1488663,0.464029,80.0,1996.0,"['17544', '9442983', '13586799', '1755629', '6..."
2,89588,0.475081,512.0,2003.0,"['114555', '219231', '455901', '102888', '6421..."
3,89583,0.465036,544.0,1999.0,"['673024', '6747831', '6056679', '121611', '65..."
4,2741850,0.457878,523.0,2008.0,"['6547188', '4894646', '1704159', '28650', '66..."


### I just realized...

This will be easier if I use lists to store the genre, author, and similar_books info. similar books is easy, just put it back in the database. But, genres was never in the form of a list and authors had a weird format so I'll have to remake those collumns, so let's do that now and add those 2 collumns to recomendations_df. 

Actually, no. For similar books, I will be looking at a book and then adding a score to each book listed under similar books. But, for authors I will be looking at each book that author has written so I will want a list of books for each author and similarly I will want a list of books for each genre. 

In [75]:
genres_list_df = (
    genres_df
    .groupby('book_id')['genres']
    .agg(list)
    .reset_index()
)

genres_list_df.head()

,book_id,genres
0,1,"[fantasy, paranormal, young-adult, fiction, ch..."
1,2,"[fantasy, paranormal, children, fiction, young..."
2,3,"[fantasy, paranormal, young-adult, fiction, ch..."
3,4,"[fantasy, paranormal, young-adult, fiction, ch..."
4,6,"[fantasy, paranormal, children, young-adult, f..."


In [12]:
genres_grouped_df = (
    genres_df
    .groupby('genres')['book_id']
    .agg(list)
    .reset_index()
)

genres_grouped_df.head()

,genres,book_id
0,biography,"[780917, 27693272, 30971685, 13579397, 1281458..."
1,children,"[1488663, 10806008, 16132671, 12814582, 290934..."
2,comics,"[22889758, 14976, 6775624, 6775623, 11365223, ..."
3,crime,"[89588, 2741850, 10806008, 27693272, 30971685,..."
4,fantasy,"[780917, 1488663, 89588, 89583, 2741850, 10806..."


In [13]:
authors_list_df = (
    authors_to_books_df
    .groupby('book_id')['author_id']
    .agg(list)
    .reset_index()
)

authors_list_df['book_id'] = authors_list_df['book_id'].astype(int)

authors_list_df.head()

,book_id,author_id
0,1,"[1077326.0, 2927.0]"
1,2,"[1077326.0, 2927.0]"
2,3,"[1077326.0, 2927.0]"
3,4,[1077326.0]
4,6,"[1077326.0, 2927.0]"


In [14]:
authors_grouped_df = (
    authors_to_books_df
    .groupby('author_id')['book_id']
    .agg(list)
    .reset_index()
)

authors_grouped_df['author_id'] = authors_grouped_df['author_id'].astype(int)

authors_grouped_df.head()

,author_id,book_id
0,4,"[6507988, 8706, 8709, 372299, 985159, 9813826,..."
1,10,"[15216, 14833682]"
2,16,"[1696917, 6254610]"
3,20,[19983]
4,23,"[19911, 19910, 19912, 19914, 19916, 6294228, 3..."


In [15]:
recommendations_df.head()

,book_id,recommendation_score,num_pages,publication_year,similar_books
0,780917,0.409143,320.0,2007.0,"['1616595', '2356304', '5432871', '9766146', '..."
1,1488663,0.464029,80.0,1996.0,"['17544', '9442983', '13586799', '1755629', '6..."
2,89588,0.475081,512.0,2003.0,"['114555', '219231', '455901', '102888', '6421..."
3,89583,0.465036,544.0,1999.0,"['673024', '6747831', '6056679', '121611', '65..."
4,2741850,0.457878,523.0,2008.0,"['6547188', '4894646', '1704159', '28650', '66..."


In [60]:
authors_grouped_df.to_csv('data/authors_grouped.csv', index='False')
genres_grouped_df.to_csv('data/genres_grouped.csv', index='False')
authors_list_df.to_csv('data/authors_list.csv', index='False')
genres_list_df.to_csv('data/genres_list.csv', index='False')
recommendations_df.to_csv('data/recommendations.csv', index='False')

# Now let's make the functions

In [18]:
def update_authors(df, authors, score):
    for author in authors:
        books = authors_grouped_df['book_id'][authors_grouped_df['author_id'] == author].values[0]
            
        for book_id in books:
            df.loc[
                df['book_id'] == book_id,
                'recommendation_score'
                ] += score

    return df

In [19]:
def update_genres(df, genres, score):
    for genre in genres:
        books = genres_grouped_df['book_id'][genres_grouped_df['genres'] == genre].values[0]

        for book_id in books:
            df.loc[
                df['book_id'] == book_id,
                'recommendation_score'
                ] += score
                
    return df

In [78]:
def update_similar_books(df, books, score):
    df = df[~df['book_id'].isin(books)]
    
    for book_id in books:
        similar_books = recommendations_df.loc[
            recommendations_df['book_id'] == book_id, 'similar_books'
        ].values[0]

        for similar_book in similar_books:
            df.loc[
                df['book_id'] == similar_book,
                'recommendation_score'
                ] += score
            
    return df

In [55]:
def update_pages(df, p1, p2):
    df = df[(df['num_pages']>=p1) & (df['num_pages']<=p2)]
    return df

In [67]:
def update_years(df, y1, y2):
    df = df[(df['publication_year']>=y1) & (df['publication_year']<=y2)]
    return df

In [ ]:
# Function to 
# prefered_authors, prefered_genres, prefered_books, etc

def get_recomendation_score(authors=[], genres=[], books=[], pages=[], years=[]):
    personalized_recommendations_df = recommendations_df.copy()

    if authors:
        n = len(authors)
        score = 1/(n/2)
        personalized_recommendations_df = update_authors(personalized_recommendations_df, authors, score)
    
    if genres:
        n = len(genres)
        score = 1/n
        personalized_recommendations_df = update_genres(personalized_recommendations_df, genres, score)

    if books:
        n = len(books)
        score = 1/n
        personalized_recommendations_df = update_similar_books(personalized_recommendations_df, books, score)

    if len(pages)==2:
        p1 = min(pages)
        p2 = max(pages)
        personalized_recommendations_df = update_pages(personalized_recommendations_df, p1, p2)

    if len(years)==2:
        y1 = min(years)
        y2 = max(years)
        personalized_recommendations_df = update_years(personalized_recommendations_df, y1, y2)

    return personalized_recommendations_df

In [57]:
def recommend_books(authors=[], genres=[], books=[], pages=[], years=[], x=10):
    # Get the personalized_recommendations_df
    personalized_recommendations_df = get_recomendation_score(authors=authors, genres=genres, books=books, pages=pages, years=years)
    
    # Get the top scores in the df
    top_df = personalized_recommendations_df.nlargest(x, 'recommendation_score')[['book_id', 'recommendation_score']].reset_index(drop=True)
    
    # Add title and ratings to df
    top_df = top_df.merge(books_df[['book_id', 'title', 'average_rating', 'ratings_count']], on='book_id', how='left')
    
    # add authors to df
    top_df["authors"] = None
    
    for i in range(x):
        entry = ""
        book_id = top_df.loc[i, 'book_id']
        auth = authors_to_books_df[authors_to_books_df['book_id']==book_id].reset_index(drop=True)
        rows = auth.shape[0]
        for row in range(rows):
            author_id = auth.loc[row, "author_id"]
            author_name = authors_df.loc[
                authors_df['author_id']==author_id,
                "name"
            ].values[0]
            role = auth.loc[row, "role"]

            entry += author_name
            if pd.notna(role):
                entry += f"({role})"
            
            if row != (rows-1):
                entry += ", "
        top_df.loc[i, "authors"] = entry

    # Add publisher, year written, number of pages, url, and image url
    top_df = top_df.merge(books_df[['book_id', 'publisher', 'publication_year', 'num_pages', 'url', 'image_url']], on='book_id', how='left')

    # Add genres
    top_df = top_df.merge(genres_list_df[['book_id', 'genres']], on='book_id', how='left')

    # Add description
    top_df = top_df.merge(books_df[['book_id', 'description']], on='book_id', how='left')

    return top_df

In [35]:
authors_to_books_df.head()

,book_id,author_id,role
0,780917,410348.0,NaN
1,1488663,238787.0,NaN
2,89588,20560.0,NaN
3,89583,20560.0,NaN
4,2741850,7581.0,NaN


Info I want to print:

* Title
* ratings
* Number of ratings
* authors and their roles[author (role), ...]
* publisher
* Year written
* number of pages
* url
* image_url
* genres
* description

books_limited:
- 'text_reviews_count'    
- 'average_rating'  
- 'similar_books' (with causion as the books are stored in a list)  
- 'num_pages' (may only be used at the end as a final filter, similar to description)  
- 'publication_year'  
- **'book_id'**  (primary key)
- 'ratings_count'  
- 'title' (to see which books are discussed)  
- 'title_without_series' (to see which books are discussed)  

books:
- 'isbn'  
- 'series'  
- 'popular_shelves'  
- 'asin'  
- 'kindle_asin'  
- 'description' 
- 'format'  
- 'link' 
- 'publisher'  
- 'publication_day'  
- 'isbn13'  
- 'publication_month'  
- 'edition_information'  
- 'url'  
- 'image_url'  
- 'work_id'

In [71]:
authors_df.head()

,average_rating,author_id,text_reviews_count,name,ratings_count
0,3.92,10333,5075,Barbara Hambly,122118
1,3.68,9212,36262,Jennifer Weiner,888522
2,4.18,19158,486,Rachel Roberts,13677
3,3.95,242185,2906,Carolyn Haines,42549
4,4.01,3389,367487,Stephen King,10666719


books:
- 'isbn'  
- 'series'  
- 'popular_shelves'  
- 'asin'  
- 'kindle_asin'  
- 'description' 
- 'format'  
- 'link' 
- 'publisher'  
- 'publication_day'  
- 'isbn13'  
- 'publication_month'  
- 'edition_information'  
- 'url'  
- 'image_url'  
- 'work_id'

In [32]:
books_df.loc[0, 'link']

'https://www.goodreads.com/book/show/780917.The_Serpent_and_the_Rose'

In [33]:
books_df.loc[0, 'url']

'https://www.goodreads.com/book/show/780917.The_Serpent_and_the_Rose'

In [34]:
books_df.loc[0, 'image_url']

'https://images.gr-assets.com/books/1316738274m/780917.jpg'

In [30]:
books_df[['link', 'url', 'image_url']]

,link,url,image_url
0,https://www.goodreads.com/book/show/780917.The...,https://www.goodreads.com/book/show/780917.The...,https://images.gr-assets.com/books/1316738274m...
1,https://www.goodreads.com/book/show/1488663.Th...,https://www.goodreads.com/book/show/1488663.Th...,https://s.gr-assets.com/assets/nophoto/book/11...
2,https://www.goodreads.com/book/show/89588.The_...,https://www.goodreads.com/book/show/89588.The_...,https://images.gr-assets.com/books/1372411409m...
3,https://www.goodreads.com/book/show/89583.Six_...,https://www.goodreads.com/book/show/89583.Six_...,https://s.gr-assets.com/assets/nophoto/book/11...
4,https://www.goodreads.com/book/show/2741850-pe...,https://www.goodreads.com/book/show/2741850-pe...,https://images.gr-assets.com/books/1406684393m...
...,...,...,...
24592,https://www.goodreads.com/book/show/279666.Ent...,https://www.goodreads.com/book/show/279666.Ent...,https://images.gr-assets.com/books/1375603617m...
24593,https://www.goodreads.com/book/show/57064.Hamm...,https://www.goodreads.com/book/show/57064.Hamm...,https://s.gr-assets.com/assets/nophoto/book/11...
24594,https://www.goodreads.com/book/show/7715664-si...,https://www.goodreads.com/book/show/7715664-si...,https://images.gr-assets.com/books/1344263648m...
24595,https://www.goodreads.com/book/show/30112512-m...,https://www.goodreads.com/book/show/30112512-m...,https://images.gr-assets.com/books/1465911243m...


In [59]:
df = recommend_books(books=[22889758, 28187, 186074, 4502507, 1215032, 12127750, 11235712], years=[2013, 2025], x=30)
df.head()

,book_id,recommendation_score,title,average_rating,ratings_count,authors,publisher,publication_year,num_pages,url,image_url,genres,description
0,18007564,0.867887,The Martian,4.39,435440,Andy Weir,Crown,2014.0,369.0,https://www.goodreads.com/book/show/18007564-t...,https://images.gr-assets.com/books/1413706054m...,"[mystery, thriller, crime, fantasy, paranormal...","Six days ago, astronaut Mark Watney became one..."
1,8755785,0.823875,"City of Heavenly Fire (The Mortal Instruments,...",4.48,183343,Cassandra Clare,Margaret K. McElderry,2014.0,725.0,https://www.goodreads.com/book/show/8755785-ci...,https://images.gr-assets.com/books/1460477794m...,"[fantasy, paranormal, young-adult, romance, fi...",In this dazzling and long-awaited conclusion t...
2,13206828,0.822227,"Cress (The Lunar Chronicles, #3)",4.46,170191,Marissa Meyer,Feiwel & Friends,2014.0,552.0,https://www.goodreads.com/book/show/13206828-c...,https://images.gr-assets.com/books/1470057005m...,"[young-adult, fantasy, paranormal, fiction, ro...","In this third book in the Lunar Chronicles, Ci..."
3,17167166,0.821749,"Crown of Midnight (Throne of Glass, #2)",4.49,169307,Sarah J. Maas,Bloomsbury USA Childrens,2013.0,418.0,https://www.goodreads.com/book/show/17167166-c...,https://images.gr-assets.com/books/1391580481m...,"[fantasy, paranormal, romance, young-adult, fi...","""A line that should never be crossed is about ..."
4,18335634,0.817675,"Clockwork Princess (The Infernal Devices, #3)",4.59,167406,Cassandra Clare,Walker Books Ltd,2013.0,567.0,https://www.goodreads.com/book/show/18335634-c...,https://images.gr-assets.com/books/1436788488m...,"[fantasy, paranormal, young-adult, romance, fi...","Danger and betrayal, love and loss, secrets an..."


In [66]:
min_year = min(books_limited_df['publication_year'][books_limited_df['publication_year']>1000])
max_year = max(books_limited_df['publication_year'])

print(min_year, max_year)

1907.0 2030.0


ok, so min will be 1900, and max will be 2026

# Tomorrow

Tomorrow, I want to finishe the function with pages and years, then create a function which will output the top 10 or otherwise specified resulting book_ids. 

Then I also want to be able to test it, to do so I need to be able to search my database for authors and books I like for example, so I need to figure out how to do this. This will be necessary for later too anyway. 

Then I can test to see how it works for recomending books I have read and possibly fine tune it and things, but I'd like to test out what I have so far tomorrow. 

# Test it out

In [ ]:
books_limited_df.head()

,book_id,title,num_pages,publication_year,average_rating,text_reviews_count,ratings_count,title_without_series,similar_books
0,780917,"The Serpent and the Rose (War of the Rose, #1)",320.0,2007.0,3.49,54,372,"The Serpent and the Rose (War of the Rose, #1)","['1616595', '2356304', '5432871', '9766146', '..."
1,1488663,The Greatest Gift: The Original Story That Ins...,80.0,1996.0,4.05,74,226,The Greatest Gift: The Original Story That Ins...,"['17544', '9442983', '13586799', '1755629', '6..."
2,89588,The Visitor,512.0,2003.0,3.81,61,1480,The Visitor,"['114555', '219231', '455901', '102888', '6421..."
3,89583,Six Moon Dance,544.0,1999.0,3.75,58,1321,Six Moon Dance,"['673024', '6747831', '6056679', '121611', '65..."
4,2741850,Personal Demon (Women of the Otherworld #8),523.0,2008.0,4.07,39,385,Personal Demon (Women of the Otherworld #8),"['6547188', '4894646', '1704159', '28650', '66..."


# Future Upgrades:

- Debugging 
- Authors you hate so they don't get recomended
- Populatity score slider - this slider will be a scaler on the popularity score to decide whether a book's popularity should be important
